# NCAA March Madness Ensembling & Calibration (v3)

This notebook focuses on:
1. **Symmetric Data Augmentation**: Training on both (T1, T2) and (T2, T1) viewpoints.
2. **Model Ensembling**: Combining XGBoost with a Logistic Regression baseline.
3. **Probability Calibration**: Ensuring predicted probabilities are well-calibrated for Log Loss.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from sklearn.metrics import log_loss
from sklearn.preprocessing import StandardScaler
import os

DATA_PATH = './march-machine-learning-mania-2026/'
team_features = pd.read_csv('all_team_features_v2.csv')
m_tourney_results = pd.read_csv(os.path.join(DATA_PATH, 'MNCAATourneyDetailedResults.csv'))
w_tourney_results = pd.read_csv(os.path.join(DATA_PATH, 'WNCAATourneyDetailedResults.csv'))
tourney_results = pd.concat([m_tourney_results, w_tourney_results], ignore_index=True)

print("Data loaded.")

## 1. Symmetric Data Augmentation

We create two rows for every game to make the model agnostic to which team is 'Team 1'.

In [ ]:
def prepare_training_data_v3(results_df, features_df):
    training_rows = []
    feat_dict = features_df.set_index(['Season', 'TeamID']).to_dict('index')
    cols_to_diff = ['SeedInt', 'OffEff', 'DefEff', 'SOS_Off', 'SOS_Def', 'TrendOffEff', 'TrendScore', 'AvgRank']
    
    for i, row in results_df.iterrows():
        season = row['Season']
        w_team, l_team = row['WTeamID'], row['LTeamID']
        
        try:
            wf = feat_dict[(season, w_team)]
            lf = feat_dict[(season, l_team)]
            
            # Row 1: Winner as Team 1 (Label 1)
            diffs1 = {f'{col}Diff': wf[col] - lf[col] for col in cols_to_diff}
            diffs1.update({'Season': season, 'label': 1})
            training_rows.append(diffs1)
            
            # Row 2: Loser as Team 1 (Label 0)
            diffs2 = {f'{col}Diff': lf[col] - wf[col] for col in cols_to_diff}
            diffs2.update({'Season': season, 'label': 0})
            training_rows.append(diffs2)
        except KeyError:
            continue
            
    return pd.DataFrame(training_rows)

train_df_aug = prepare_training_data_v3(tourney_results, team_features)
print(f"Augmented training dataset created with {len(train_df_aug)} rows.")

## 2. Ensemble: XGBoost + Logistic Regression

Logistic Regression helps calibrate the 'obvious' features like Seed difference, while XGBoost captures non-linear interactions.

In [ ]:
X = train_df_aug.drop(['Season', 'label'], axis=1)
y = train_df_aug['label']

# Scaling is required for Logistic Regression
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

params_xgb = {
    'eval_metric': 'logloss', 'max_depth': 4, 'learning_rate': 0.05, 
    'n_estimators': 200, 'subsample': 0.8, 'colsample_bytree': 0.8, 'random_state': 42
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)
xgb_total_scores, lr_total_scores, ensemble_total_scores = [], [], []
alpha = 0.6 # Weight for XGBoost

for train_idx, val_idx in kf.split(X):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    X_train_scaled, X_val_scaled = X_scaled[train_idx], X_scaled[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # XGBoost
    m_xgb = xgb.XGBClassifier(**params_xgb)
    m_xgb.fit(X_train, y_train)
    p_xgb = m_xgb.predict_proba(X_val)[:, 1]
    
    # Logistic Regression
    m_lr = LogisticRegression()
    m_lr.fit(X_train_scaled, y_train)
    p_lr = m_lr.predict_proba(X_val_scaled)[:, 1]
    
    # Ensemble
    p_ensemble = alpha * p_xgb + (1 - alpha) * p_lr
    
    xgb_total_scores.append(log_loss(y_val, p_xgb))
    lr_total_scores.append(log_loss(y_val, p_lr))
    ensemble_total_scores.append(log_loss(y_val, p_ensemble))

print(f"XGB Loss: {np.mean(xgb_total_scores):.4f}")
print(f"LR Loss:  {np.mean(lr_total_scores):.4f}")
print(f"Ensemble Loss: {np.mean(ensemble_total_scores):.4f}")

## 3. Submission with Calibration

We clip the probabilities to [0.025, 0.975] to avoid massive Log Loss penalties from 'certain' but wrong predictions.

In [ ]:
# Pre-train final models
final_xgb = xgb.XGBClassifier(**params_xgb)
final_xgb.fit(X, y)

final_lr = LogisticRegression()
final_lr.fit(X_scaled, y)

def predict_matchup_v3(season, t1, t2, feat_dict, xgb_model, lr_model, scaler):
    try:
        f1 = feat_dict[(season, t1)]
        f2 = feat_dict[(season, t2)]
        
        cols = ['SeedInt', 'OffEff', 'DefEff', 'SOS_Off', 'SOS_Def', 'TrendOffEff', 'TrendScore', 'AvgRank']
        feat_row = pd.DataFrame([{f'{col}Diff': f1[col] - f2[col] for col in cols}])
        
        p_xgb = xgb_model.predict_proba(feat_row)[:, 1][0]
        
        feat_scaled = scaler.transform(feat_row)
        p_lr = lr_model.predict_proba(feat_scaled)[:, 1][0]
        
        p = 0.6 * p_xgb + 0.4 * p_lr
        return np.clip(p, 0.025, 0.975)
    except KeyError:
        return 0.5

submission = pd.read_csv(os.path.join(DATA_PATH, 'SampleSubmissionStage1.csv'))
feat_lookup = team_features.set_index(['Season', 'TeamID']).to_dict('index')

submission['Pred'] = submission.apply(lambda r: predict_matchup_v3(int(r['ID'].split('_')[0]), int(r['ID'].split('_')[1]), int(r['ID'].split('_')[2]), feat_lookup, final_xgb, final_lr, scaler), axis=1)
submission.to_csv('submission_v3.csv', index=False)
print("v3 Submission (Ensemble + Clipping) complete!")